# Project Visualization Notebook

**SDS320 · L03 mini deliverable** · Project: Sentinel-2 vegetation-stress indices for drought-induced forest dieback (Valais/Jura) since the 2018 summer drought

This notebook is reused in L04 (training data) and L05 (image recognition). Everything study-area-specific sits in the **Config** cell, so you can change the AOI or years in one place.

| # | Deliverable (Project transfer, §5) | Section |
| --- | --- | --- |
| 1 | Interactive map of the study area with an appropriate basemap | 1 |
| 2 | At least one raster/vector layer from own candidate data, styled deliberately | 2–4 |
| 3 | One split-panel comparison relevant to the project question | 5 |
| 4 | 2–3 sentence note on what the visualization revealed | 7 |

Methods used from the L03 notebooks: basemaps (01), raster styling with `vmin`/`vmax`/colormap (02), Planetary Computer search + preview with `expression` (03), vector overlay with `add_gdf` and a style dict (04), `split_map` (05), transparent overlays (06).

## 0. Setup and config

In [ ]:
import warnings
from collections import Counter
from pathlib import Path

import geoai
import geopandas as gpd
import leafmap.foliumap as leafmap
import numpy as np
import planetary_computer as pc
import rasterio
from rasterio.windows import Window, bounds as window_bounds
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling
from shapely.geometry import box, shape

In [ ]:
# ---- Config: edit here -------------------------------------------------------
AOI_NAME = "Sierre - Leuk (VS)"              # placeholder AOI: Rhone valley with Scots pine slopes
AOI_BBOX = [7.45, 46.24, 7.80, 46.38]        # [west, south, east, north], EPSG:4326 (~27 x 15 km)
BEFORE_YEAR, AFTER_YEAR = 2017, 2019          # pre-drought vs. post-drought summer
SEASON = ("07-01", "08-31")                   # same season in both years -> comparable phenology
MAX_CLOUD = 20                                # max. scene cloud cover (%)
N_SCENES = 5                                  # scenes per year for the median composite
OUT_DIR = Path("../data/processed")           # notebook lives in notebooks/, data in data/
OUT_DIR.mkdir(parents=True, exist_ok=True)

CENTER = [(AOI_BBOX[1] + AOI_BBOX[3]) / 2, (AOI_BBOX[0] + AOI_BBOX[2]) / 2]  # [lat, lon]

In [ ]:
def make_map(basemap="SATELLITE", height="500px", zoom=11):
    """Reusable base map: centred on the AOI with a satellite basemap by default."""
    m = leafmap.Map(center=CENTER, zoom=zoom, height=height)
    m.add_basemap(basemap)
    return m

aoi_gdf = gpd.GeoDataFrame({"name": [AOI_NAME]}, geometry=[box(*AOI_BBOX)], crs="EPSG:4326")
AOI_STYLE = {"color": "white", "weight": 2, "fillOpacity": 0}

In [ ]:
import branca.colormap as bcm
import matplotlib as mpl


def add_legend(m, style, caption, n=10):
    """Add a colour-bar legend that matches a raster style dict (colormap, vmin, vmax)."""
    cmap = mpl.colormaps[style["colormap"]]
    colors = [mpl.colors.to_hex(cmap(x)) for x in np.linspace(0, 1, n)]
    bcm.LinearColormap(colors, vmin=style["vmin"], vmax=style["vmax"], caption=caption).add_to(m)
    return m

## 1. Study area and basemap

Forest condition is judged visually, so a **satellite basemap** is the default. The topographic map is added for aspect and elevation (south-facing slopes dry out first). Use the layer control to switch.

In [ ]:
m = make_map("SATELLITE")
m.add_basemap("SwissFederalGeoportal.SWISSIMAGE")
m.add_basemap("OpenTopoMap")
m.add_gdf(aoi_gdf, layer_name=f"AOI: {AOI_NAME}", style=AOI_STYLE)
m

**Check:** Is the map centred on the right place? Is the AOI small enough to inspect (and does it contain forest)? Adjust `AOI_BBOX` if not.

## 2. Search and preview Sentinel-2 (nothing downloaded yet)

Same pattern as the Planetary Computer notebook: search per year, check assets, preview on the fly.

In [ ]:
def search_year(year):
    return geoai.pc_stac_search(
        collection="sentinel-2-l2a",
        bbox=AOI_BBOX,
        time_range=f"{year}-{SEASON[0]}/{year}-{SEASON[1]}",
        query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
        max_items=50,
    )

def pick_items(items, n=N_SCENES, tile=None):
    """Keep the n least cloudy scenes from one MGRS tile that fully covers the AOI."""
    aoi = box(*AOI_BBOX)
    covering = [i for i in items if shape(i.geometry).contains(aoi)] or items
    if tile is None:
        tile = Counter(i.properties["s2:mgrs_tile"] for i in covering).most_common(1)[0][0]
    same = [i for i in covering if i.properties["s2:mgrs_tile"] == tile]
    return sorted(same, key=lambda i: i.properties["eo:cloud_cover"])[:n]

before_items = pick_items(search_year(BEFORE_YEAR))
TILE = before_items[0].properties["s2:mgrs_tile"]   # use the same tile (pixel grid) in both years
after_items = pick_items(search_year(AFTER_YEAR), tile=TILE)

for label, items in [("BEFORE", before_items), ("AFTER", after_items)]:
    print(label)
    for i in items:
        print(f"  {i.datetime:%Y-%m-%d}  tile {i.properties['s2:mgrs_tile']}  cloud {i.properties['eo:cloud_cover']:.1f}%")

BEFORE
  2017-08-18  tile 32TLS  cloud 2.9%
  2017-08-23  tile 32TLS  cloud 8.5%
  2017-07-19  tile 32TLS  cloud 14.9%
  2017-07-29  tile 32TLS  cloud 19.6%
AFTER
  2019-07-24  tile 32TLS  cloud 1.6%
  2019-08-18  tile 32TLS  cloud 1.9%
  2019-07-19  tile 32TLS  cloud 6.8%
  2019-08-08  tile 32TLS  cloud 10.5%
  2019-07-14  tile 32TLS  cloud 16.0%


In [ ]:
geoai.pc_item_asset_list(after_items[0])

['AOT',
 'B01',
 'B02',
 'B03',
 'B04',
 'B05',
 'B06',
 'B07',
 'B08',
 'B09',
 'B11',
 'B12',
 'B8A',
 'SCL',
 'WVP',
 'visual',
 'preview',
 'safe-manifest',
 'granule-metadata',
 'inspire-metadata',
 'product-metadata',
 'datastrip-metadata',
 'tilejson',
 'rendered_preview']

In [ ]:
# Preview 1: false colour (NIR, red, green) -> healthy vegetation bright red
# Note: streams a whole Sentinel-2 tile, so it can take a while to load
m_prev = geoai.view_pc_item(
    item=after_items[0],
    assets=["B08", "B04", "B03"],
    rescale="0,3000",  # Sentinel-2 DNs run 0-10000; without this the map stays grey
    name=f"False colour {AFTER_YEAR} (NIR-R-G)",
    map_args={"center": CENTER, "zoom": 12},
)
m_prev

In [ ]:
# Preview 2: NDVI computed server-side
m_ndvi = geoai.view_pc_item(
    item=after_items[0],
    expression="(B08-B04)/(B08+B04)",
    rescale="0,1",
    colormap_name="ylgn",
    name=f"NDVI {AFTER_YEAR} (preview)",
)
m_ndvi

## 3. Build composites for the AOI

For each year: read only the AOI window from each scene, mask clouds/shadows/snow with the scene classification layer (SCL), compute **NDVI** and **NDMI** (moisture, uses SWIR B11), and take the per-pixel **median** across scenes. Results are saved as small GeoTIFFs in `OUT_DIR`.

Note: since processing baseline 04.00 (2022) Sentinel-2 DNs carry a +1000 offset. The code corrects this so years stay comparable.

In [ ]:
SCL_BAD = [0, 1, 3, 8, 9, 10, 11]  # no data, saturated, cloud shadow, cloud medium/high, cirrus, snow


def _ref_window(src):
    """Pixel window of the AOI on the 10 m grid of an open raster."""
    b = transform_bounds("EPSG:4326", src.crs, *AOI_BBOX)
    w = src.window(*b)
    col, row = int(np.floor(w.col_off)), int(np.floor(w.row_off))
    return Window(col, row, int(np.ceil(w.width)), int(np.ceil(w.height)))


def _read_like(href, bnds, shape, resampling):
    """Read a band (any resolution) for given bounds, resampled onto the reference grid."""
    with rasterio.open(href) as src:
        w = src.window(*bnds)
        return src.read(1, window=w, out_shape=shape, resampling=resampling).astype("float32")


def indices_for_item(item):
    """Cloud-masked NDVI and NDMI for the AOI of one scene."""
    a = pc.sign(item).assets
    with rasterio.open(a["B08"].href) as src:
        win = _ref_window(src)
        nir = src.read(1, window=win).astype("float32")
        transform = src.window_transform(win)
        bnds = window_bounds(win, src.transform)
        crs = src.crs
    shape = nir.shape
    red = _read_like(a["B04"].href, bnds, shape, Resampling.bilinear)
    swir = _read_like(a["B11"].href, bnds, shape, Resampling.bilinear)
    scl = _read_like(a["SCL"].href, bnds, shape, Resampling.nearest)

    offset = 1000 if float(item.properties.get("s2:processing_baseline", "0")) >= 4.0 else 0
    nir, red, swir = [(x - offset) / 10000 for x in (nir, red, swir)]

    with np.errstate(divide="ignore", invalid="ignore"):
        ndvi = (nir - red) / (nir + red)
        ndmi = (nir - swir) / (nir + swir)
    bad = np.isin(scl, SCL_BAD) | ~np.isfinite(ndvi) | ~np.isfinite(ndmi)
    ndvi[bad], ndmi[bad] = np.nan, np.nan
    return ndvi, ndmi, {"transform": transform, "crs": crs}


def composite(items):
    nd, nm = [], []
    for it in items:
        v, mo, meta = indices_for_item(it)
        nd.append(v)
        nm.append(mo)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(np.stack(nd), 0), np.nanmedian(np.stack(nm), 0), meta


def save_tif(arr, meta, path):
    """Write float32 GeoTIFF with a finite nodata value (-9999); NaN nodata can upset the tile server."""
    data = np.where(np.isfinite(arr), arr, -9999).astype("float32")
    with rasterio.open(
        path, "w", driver="GTiff", height=data.shape[0], width=data.shape[1], count=1,
        dtype="float32", crs=meta["crs"], transform=meta["transform"], nodata=-9999, compress="deflate",
    ) as dst:
        dst.write(data, 1)
    return str(Path(path).resolve())  # absolute path for localtileserver

In [ ]:
paths = {}
arrays = {}
for year, items in [(BEFORE_YEAR, before_items), (AFTER_YEAR, after_items)]:
    ndvi, ndmi, meta = composite(items)
    arrays[year] = (ndvi, ndmi)
    paths[f"ndvi_{year}"] = save_tif(ndvi, meta, OUT_DIR / f"ndvi_{year}_median.tif")
    paths[f"ndmi_{year}"] = save_tif(ndmi, meta, OUT_DIR / f"ndmi_{year}_median.tif")
    print(f"{year}: {np.isfinite(ndvi).mean():.0%} valid pixels")

# Change layer: after minus before (negative = vegetation loss)
dndvi = arrays[AFTER_YEAR][0] - arrays[BEFORE_YEAR][0]
paths["dndvi"] = save_tif(dndvi, meta, OUT_DIR / f"dndvi_{BEFORE_YEAR}_{AFTER_YEAR}.tif")
paths

2017: 100% valid pixels
2019: 99% valid pixels


{'ndvi_2017': '/Users/davidleu/Desktop/sds320-project/data/processed/ndvi_2017_median.tif',
 'ndmi_2017': '/Users/davidleu/Desktop/sds320-project/data/processed/ndmi_2017_median.tif',
 'ndvi_2019': '/Users/davidleu/Desktop/sds320-project/data/processed/ndvi_2019_median.tif',
 'ndmi_2019': '/Users/davidleu/Desktop/sds320-project/data/processed/ndmi_2019_median.tif',
 'dndvi': '/Users/davidleu/Desktop/sds320-project/data/processed/dndvi_2017_2019.tif'}

In [ ]:
# QC: are both composites valid and on the same footprint?
for key in [f"ndvi_{BEFORE_YEAR}", f"ndvi_{AFTER_YEAR}"]:
    with rasterio.open(paths[key]) as src:
        a = src.read(1, masked=True)
        print(key, "| valid:", f"{a.count() / a.size:.0%}", "| min/max:", a.min(), a.max(), "| bounds:", src.bounds)

ndvi_2017 | valid: 100% | min/max: -0.8461539 0.99529403 | bounds: BoundingBox(left=380500.0, bottom=5121410.0, right=407720.0, top=5137440.0)
ndvi_2019 | valid: 99% | min/max: -0.92307687 0.9827089 | bounds: BoundingBox(left=380500.0, bottom=5121410.0, right=407720.0, top=5137440.0)


**Check:** Low valid-pixel share means too many clouds/snow. Raise `MAX_CLOUD`, `N_SCENES` or widen `SEASON`.

## 4. Styled single layers on the map

Deliberate styling: **sequential** colormap for NDVI (continuous, low → high), **diverging** colormap for the change layer (meaningful centre at 0), fixed value ranges, descriptive layer names, and partial opacity so the imagery stays visible. No rainbow colormaps.

In [ ]:
NDVI_STYLE = {"vmin": 0.2, "vmax": 0.9, "colormap": "YlGn"}
DIFF_STYLE = {"vmin": -0.3, "vmax": 0.3, "colormap": "RdBu"}

m2 = make_map("SATELLITE")
m2.add_raster(paths[f"ndvi_{AFTER_YEAR}"], layer_name=f"NDVI {AFTER_YEAR} (median Jul-Aug)", opacity=0.8, **NDVI_STYLE)
m2.add_raster(paths["dndvi"], layer_name=f"dNDVI {BEFORE_YEAR} to {AFTER_YEAR} (red = loss)", opacity=0.8, **DIFF_STYLE)
m2.add_gdf(aoi_gdf, layer_name="AOI", style=AOI_STYLE)
add_legend(m2, NDVI_STYLE, "NDVI (low = bare/sparse, high = dense green vegetation)")
add_legend(m2, DIFF_STYLE, "dNDVI (red = vegetation loss, blue = gain)")
m2

**Optional vector layer.** If you have candidate damage/forest polygons from L02 (e.g. documented dieback areas, forest mask), put them at the path below. They are drawn with a thin outline and low fill so imagery and index stay visible. Overlay them *before* trusting them as labels.

In [ ]:
VECTOR_PATH = Path("../data/raw/dieback_polygons.geojson")   # <- replace with your own file

if VECTOR_PATH.exists():
    labels = gpd.read_file(VECTOR_PATH)
    print("CRS:", labels.crs, "| features:", len(labels), "|", labels.geom_type.value_counts().to_dict())
    labels = labels.to_crs("EPSG:4326")
    m3 = make_map("SATELLITE")
    m3.add_raster(paths["dndvi"], layer_name=f"dNDVI {BEFORE_YEAR} to {AFTER_YEAR}", opacity=0.7, **DIFF_STYLE)
    m3.add_gdf(labels, layer_name="Documented damage (candidate labels)",
               style={"color": "white", "weight": 2, "fillColor": "white", "fillOpacity": 0.1})
    display(m3)
else:
    print(f"No vector file at {VECTOR_PATH} - skipping. Add one to check label alignment.")

No vector file at ../data/raw/dieback_polygons.geojson - skipping. Add one to check label alignment.


## 5. Split-panel comparison: before vs. after drought

Question: *Where did vegetation vigour drop between the pre-drought and the post-drought summer?* Both sides use the **same colormap and value range**, so any difference you see is in the data, not the styling.

**Reading the legend:** a patch that moves from dark green (left) to pale yellow (right) lost vegetation vigour between the two years.

In [ ]:
# Warm-up: serve each file once via add_raster; split_map can fail on the first request otherwise
_w = make_map("SATELLITE")
for k in (f"ndvi_{BEFORE_YEAR}", f"ndvi_{AFTER_YEAR}"):
    _w.add_raster(paths[k], layer_name=k, **NDVI_STYLE)

m4 = make_map("SATELLITE")
m4.split_map(
    left_layer=paths[f"ndvi_{BEFORE_YEAR}"],
    right_layer=paths[f"ndvi_{AFTER_YEAR}"],
    left_args=dict(NDVI_STYLE),
    right_args=dict(NDVI_STYLE),
    left_label=f"NDVI {BEFORE_YEAR} (before)",
    right_label=f"NDVI {AFTER_YEAR} (after)",
)
add_legend(m4, NDVI_STYLE, "NDVI (low = bare/sparse, high = dense green vegetation)")
m4

In [ ]:
# Same pattern for moisture (NDMI) - often more sensitive to water stress than NDVI
NDMI_STYLE = {"vmin": -0.2, "vmax": 0.6, "colormap": "YlGnBu"}
_w = make_map("SATELLITE")
for k in (f"ndmi_{BEFORE_YEAR}", f"ndmi_{AFTER_YEAR}"):
    _w.add_raster(paths[k], layer_name=k, **NDMI_STYLE)

m5 = make_map("SATELLITE")
m5.split_map(
    left_layer=paths[f"ndmi_{BEFORE_YEAR}"],
    right_layer=paths[f"ndmi_{AFTER_YEAR}"],
    left_args=dict(NDMI_STYLE),
    right_args=dict(NDMI_STYLE),
    left_label=f"NDMI {BEFORE_YEAR} (before)",
    right_label=f"NDMI {AFTER_YEAR} (after)",
)
add_legend(m5, NDMI_STYLE, "NDMI (low = dry, high = moist vegetation)")
m5

## 6. Quick numbers to support what you see

In [ ]:
valid = np.isfinite(dndvi)
print(f"Valid pixels: {valid.mean():.0%}")
print(f"Mean NDVI {BEFORE_YEAR}: {np.nanmean(arrays[BEFORE_YEAR][0]):.3f} | {AFTER_YEAR}: {np.nanmean(arrays[AFTER_YEAR][0]):.3f}")
print(f"Pixels with dNDVI < -0.1: {(dndvi[valid] < -0.1).mean():.1%}  (all land cover, not only forest!)")

Valid pixels: 99%
Mean NDVI 2017: 0.588 | 2019: 0.595
Pixels with dNDVI < -0.1: 3.6%  (all land cover, not only forest!)


## 7. Note: what the visualization revealed

*(2–3 sentences - fill in after running; mention anything wrong or unexpected.)*

> TODO: e.g. "NDVI drops mainly on south-facing slopes below ... m. Some strong negative values are clearly not forest (fields, snow, cloud edges). The `...` scene still shows haze, so the composite needs ..."

**Things to look for**
- Cloud/shadow/snow remnants showing up as false change
- Non-forest change (harvest, agriculture, construction) mixed into dNDVI
- Mountain shadows and terrain effects on NDVI
- Misalignment of any candidate labels with the imagery

## 8. Self-check (best practices) and L03 checklist

| Check | Status |
| --- | --- |
| Colormap matches data type (sequential NDVI/NDMI, diverging dNDVI, no rainbow) | done in sections 4–5 |
| Spatial context (satellite basemap + AOI outline) | done |
| Consistent styling in comparisons (same colormap and range left/right) | done in section 5 |
| Descriptive layer names | done |
| Audience: currently "me during analysis"; for the slides, reduce to one layer plus legend and caption | TODO |

- [ ] Own candidate data on an interactive map
- [ ] Labels checked against imagery (needs the optional vector layer)
- [ ] Band combination / colormap chosen on what I actually saw
- [ ] Split-panel comparison built
- [ ] Best-practices self-check applied

**Next (L04):** reuse `make_map()` and `composite()` to overlay training tiles/labels on these layers.